# TP21 — Orchestration & outils

**25 novembre · jalon J5 · ½ journée après le module 6.4**

## Objectif (énoncé du TP)

Un assistant **multi-tours** qui combine RAG, base produits et substitution ML :

1. reprendre la chaîne dans **LangChain** : gestion de la conversation, **mémoire courte** ;
2. déclarer trois outils : `chercher_produit` (TP18), `substituer` (TP16), `repondre_source` (TP19-20) — le modèle choisit l'outil selon la question ;
3. faire passer le scénario : *ce produit est-il bon ? → pourquoi ? → propose mieux → et en bio ?* ;
4. **journaliser** chaque échange : outil appelé, latence, tokens ;
5. revue croisée de 10 minutes avec une autre équipe, remarques en issues.

**À committer** : assistant orchestré + scénarios rejouables + journalisation active.

## Plan du notebook

| Étape | Contenu | Point de l'énoncé |
|---|---|---|
| 0 | Décisions et environnement | — |
| 1 | Session (état de la conversation) | 1 |
| 2 | Services injectables + fixtures | 2 |
| 3 | Schémas et outils LangChain | 2 |
| 4 | Mémoire courte | 1 |
| 5 | Journalisation (traces) | 4 |
| 6 | Boucle d'orchestration | 1, 2 |
| 7 | Tests sans modèle (modèle scripté + fixtures) | 3 |
| 8 | Scénario rejouable + rapport | 3 |
| 9 | Brancher les services réels | 2 |
| 10 | Rejeu avec le vrai modèle | 3, 4 |
| 11 | Cas de contrôle | — |
| 12 | Revue croisée et rendu | 5 |

## Comment travailler

- Les cellules marquées **TODO** sont à écrire par toi. Elles lèvent `NotImplementedError` tant qu'elles ne sont pas faites.
- Les cellules **Test** utilisent `assert` : tant qu'elles échouent, l'étape n'est pas terminée.
- Les aides sont repliées : essaie d'abord seul, puis ouvre **Aide +1**, puis **+2**, puis **+3** seulement si tu bloques.
- Les **fixtures** sont des données **fictives** pour tester la plomberie. Un parcours simulé valide le routage, pas la qualité de l'assistant.

## Étape 0 — Décisions et environnement

**Base d'intégration (à valider avec ton binôme avant de créer la branche).** Ton analyse précédente recommande de partir de `develop` et de prendre la PR #71 comme base technique : outil produit, retrieval Chroma (`rechercher`) et jeu d'évaluation. À vérifier : relis les diffs des PR #70 et #71 contre `develop` au moment d'intégrer, et arbitre explicitement les fichiers `src/assistant/*` communs.

**Choix d'architecture de ce notebook (à savoir défendre à l'oral).**

- **Boucle d'outils écrite à la main** avec `bind_tools` (LangChain) plutôt qu'un agent prêt à l'emploi : on voit chaque appel d'outil, on mesure latence et tokens, et le code reste indépendant de la version de LangChain. Alternative à mentionner : `create_agent` / `AgentExecutor`, plus court mais plus opaque pour tracer.
- **Session explicite** : le code du produit confirmé vit dans l'objet `Session`, pas dans le texte des messages (qui sont tronqués par la mémoire courte).
- **Services injectés** : les outils ne connaissent ni le catalogue ni Chroma directement, ce qui permet de tester avec des fixtures puis de brancher le réel.

**À faire d'abord** : note la version de LangChain vue au module 6.4 et vérifie que le noyau a bien les paquets (ta dernière exécution indiquait que `langchain` n'était pas disponible dans le noyau).

In [7]:
import json
import sys
import time
from dataclasses import asdict, dataclass, field
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Callable, Optional

print('Python :', sys.version.split()[0])
for paquet in ['langchain', 'langchain-core', 'langchain-ollama', 'pydantic']:
    try:
        print(f'{paquet} : {version(paquet)}')
    except PackageNotFoundError:
        print(f'{paquet} : NON INSTALLÉ  →  pip install {paquet}')

# TODO 0 : recopie ici la version de LangChain du module 6.4 : ...


Python : 3.12.10
langchain : 1.5.2
langchain-core : 1.8.4
langchain-ollama : 1.1.0
pydantic : 2.14.3


In [8]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import (
    BaseModel, ConfigDict, Field, StrictBool, StrictInt, ValidationError, model_validator,
)

# --- Paramètres (à adapter) ---
MODELE = 'qwen3:8b'                 # TODO : mets le modèle réellement utilisé au TP18
URL_OLLAMA = 'http://localhost:11434'
MAX_ITERATIONS_OUTILS = 4           # garde-fou : nombre max d'allers-retours modèle/outils par tour
NB_ECHANGES_MEMOIRE = 3             # mémoire courte : nombre d'échanges précédents conservés
FICHIER_TRACES = Path('logs') / 'traces_tp21.jsonl'  # TODO : aligne avec l'arborescence du dépôt

## Étape 1 — La session

Une **session** regroupe l'état d'une conversation : identifiant, historique des messages, numéro de tour et **code du produit confirmé**.

Le code confirmé est ce qui permet à « Pourquoi ? » ou « Et en bio ? » de savoir de quel produit on parle, sans que le modèle ait à le deviner.

**Point de contrôle (réponds par écrit dans la cellule suivante).** Où vit le code du produit confirmé, et à quel moment change-t-il ? Que se passe-t-il s'il y a plusieurs candidats (recherche ambiguë) ?

In [9]:
@dataclass
class Session:
    '''État d'une conversation avec l'assistant.'''
    session_id: str
    code_confirme: Optional[str] = None            # code-barres, toujours une chaîne
    historique: list = field(default_factory=list)  # messages LangChain, tours précédents
    numero_tour: int = 0


# TODO 1 (réponse écrite, 3 à 5 lignes) :
# - où vit le code du produit confirmé ?
# - quand est-il modifié ? quand ne doit-il surtout PAS être modifié ?
# - que se passe-t-il quand une nouvelle session démarre ?
"""Le code du produit confirmé est stocké dans `Session.code_confirme`.
Il est rempli uniquement après une recherche exacte réussie : un seul produit doit être trouvé.
Il reste inchangé pour « Pourquoi ? » et « Et en bio ? », afin d’éviter qu’ils changent de produit.
Il ne doit pas être modifié par les autres outils ou par une recherche ambiguë ou introuvable.
Une nouvelle session crée une nouvelle instance `Session` avec `code_confirme = None`.
Il faut donc demander une confirmation ou un code-barres dès qu’on commence."""


'Le code du produit confirmé est stocké dans `Session.code_confirme`.\nIl est rempli uniquement après une recherche exacte réussie : un seul produit doit être trouvé.\nIl reste inchangé pour « Pourquoi ? » et « Et en bio ? », afin d’éviter qu’ils changent de produit.\nIl ne doit pas être modifié par les autres outils ou par une recherche ambiguë ou introuvable.\nUne nouvelle session crée une nouvelle instance `Session` avec `code_confirme = None`.\nIl faut donc demander une confirmation ou un code-barres dès qu’on commence.'

## Étape 2 — Services injectables et fixtures

Un `Services` regroupe trois fonctions : chercher un produit, substituer, retrouver des sources. Chacune prend **un dictionnaire d'arguments** et renvoie **un dictionnaire** (contrats de l'étape 3).

Pour pouvoir avancer sans dépendre du catalogue réel, la cellule suivante fournit des **fixtures FICTIVES**. Les vrais services seront branchés à l'étape 9.

Regarde bien le format des résultats : c'est le contrat que les vrais services devront respecter.

In [10]:
# ⚠️ FIXTURES : données FICTIVES pour tester la plomberie (routage, session, traces).
ORDRE_GRADES = 'abcde'
CODE_FIXTURE = '9990000000001'

CATALOGUE_FIXTURE = [
    {'code': '9990000000001', 'nom': 'Produit test A', 'marque': 'Marque test', 'nutriscore': 'e',
     'nutriments': {'sugars': 50.0, 'salt': 0.1}, 'bio': False, 'rayon': 'rayon test'},
    {'code': '9990000000002', 'nom': 'Produit test B', 'marque': 'Marque test', 'nutriscore': 'c',
     'nutriments': {'sugars': 20.0, 'salt': 0.2}, 'bio': False, 'rayon': 'rayon test'},
    {'code': '9990000000003', 'nom': 'Produit test C bio', 'marque': 'Marque bio', 'nutriscore': 'b',
     'nutriments': {'sugars': 10.0, 'salt': 0.1}, 'bio': True, 'rayon': 'rayon test'},
    {'code': '9990000000004', 'nom': 'Produit test D', 'marque': 'Marque test', 'nutriscore': 'a',
     'nutriments': {'sugars': 5.0, 'salt': 0.0}, 'bio': False, 'rayon': 'rayon test'},
    {'code': '9990000000005', 'nom': 'Produit test E', 'marque': 'Marque test', 'nutriscore': None,
     'nutriments': {'sugars': None, 'salt': None}, 'bio': None, 'rayon': 'rayon test'},
]


def _fiche(produit: dict) -> dict:
    '''Met un produit au format fiche : les inconnues restent à None.'''
    return {
        'code': produit['code'], 'nom': produit['nom'], 'marque': produit['marque'],
        'base': 'fixture', 'nutriscore': produit['nutriscore'],
        'nutriments': produit['nutriments'],
        'donnees_incompletes': produit['nutriscore'] is None,
    }


def chercher_fixture(arguments: dict) -> dict:
    code = arguments.get('code')
    nom = (arguments.get('nom') or '').lower()
    limite = arguments.get('limite') or 3
    if code:
        trouves = [p for p in CATALOGUE_FIXTURE if p['code'] == code]
    else:
        trouves = [p for p in CATALOGUE_FIXTURE if nom and nom in p['nom'].lower()]
    if not trouves:
        return {'statut': 'introuvable', 'total': 0, 'produits': []}
    statut = 'ok' if len(trouves) == 1 else 'ambigu'
    return {'statut': statut, 'total': len(trouves), 'produits': [_fiche(p) for p in trouves[:limite]]}


def substituer_fixture(arguments: dict) -> dict:
    code = arguments.get('code')
    ref = next((p for p in CATALOGUE_FIXTURE if p['code'] == code), None)
    if ref is None:
        return {'statut': 'introuvable', 'code_reference': code, 'substituts': []}
    if ref['nutriscore'] is None:
        return {'statut': 'donnees_insuffisantes', 'code_reference': code, 'substituts': []}
    rang_ref = ORDRE_GRADES.index(ref['nutriscore'])
    candidats = [
        p for p in CATALOGUE_FIXTURE
        if p['code'] != code and p['rayon'] == ref['rayon'] and p['nutriscore'] is not None
        and ORDRE_GRADES.index(p['nutriscore']) < rang_ref
    ]
    if arguments.get('bio_uniquement'):
        candidats = [p for p in candidats if p['bio'] is True]
    candidats.sort(key=lambda p: (ORDRE_GRADES.index(p['nutriscore']), p['code']))
    limite = arguments.get('limite') or 3
    substituts = [{'produit': _fiche(p), 'bio': p['bio'], 'distance': None} for p in candidats[:limite]]
    statut = 'ok' if substituts else 'aucun_substitut'
    return {'statut': statut, 'code_reference': code, 'substituts': substituts}


def repondre_source_fixture(arguments: dict) -> dict:
    code = arguments.get('code')
    produit = next((p for p in CATALOGUE_FIXTURE if p['code'] == code), None)
    if produit is None:
        return {'statut': 'sources_insuffisantes', 'reponse': 'Aucune source disponible.', 'sources': []}
    grade = produit['nutriscore']
    return {
        'statut': 'ok',
        'reponse': f'[FIXTURE] Le produit {code} a le grade {grade} dans la fiche fictive.',
        'sources': [{'id': f'fixture-fiche-{code}', 'titre': 'Fiche catalogue (fixture)',
                     'url': None, 'extrait': f'grade {grade}'}],
    }


@dataclass
class Services:
    '''Dépendances injectées dans les outils.'''
    chercher: Callable[[dict], dict]
    substituer: Callable[[dict], dict]
    rechercher_sources: Callable[[dict], dict]
    simule: bool = True   # True = fixtures : le rejeu ne prouve que le routage


SERVICES_FIXTURE = Services(chercher_fixture, substituer_fixture, repondre_source_fixture, simule=True)
print(chercher_fixture({'code': CODE_FIXTURE})['statut'])

ok


## Étape 3 — Schémas et outils LangChain

Le modèle ne voit de chaque outil que **son nom, sa description et le schéma de ses arguments**. C'est la description qui lui permet de choisir entre `substituer` et `repondre_source`.

Règles de contrat (reprises de ton analyse) : schémas qui **refusent les champs inconnus** (`extra='forbid'`), entiers **stricts** pour `limite` et `k`, codes-barres en **chaînes**, erreurs de validation distinguées des recherches sans résultat.

`ArgsChercherProduit` est fourni en exemple complet. À toi de faire `ArgsSubstituer` et `ArgsRepondreSource`, puis de les brancher.

| Outil | Arguments | Résultat |
|---|---|---|
| `chercher_produit` | `nom`, `code`, `limite` (1–5, défaut 3) ; au moins un nom ou un code | `{statut, total, produits}` — statut `ok`, `ambigu`, `introuvable` |
| `substituer` | `code`, `bio_uniquement` (bool, défaut False), `limite` (1–5, défaut 3) | `{statut, code_reference, substituts}` — statut `ok`, `introuvable`, `donnees_insuffisantes`, `aucun_substitut` |
| `repondre_source` | `question` (non vide), `code` (optionnel), `k` (1–20, défaut 5) | `{statut, reponse, sources}` — statut `ok`, `sources_insuffisantes` |

En cas d'arguments invalides : `{erreur}`.

<details><summary>Aide +1 (question)</summary>

Le modèle peut oublier ou inventer un `code`. Pour `substituer` et `repondre_source`, qui est la source fiable du produit de référence : l'argument du modèle ou la session ?
</details>

<details><summary>Aide +2 (approche)</summary>

Rends `code` **optionnel** dans le schéma et complète-le avec `session.code_confirme` dans la fonction de l'outil. Attention : lis `session.code_confirme` **au moment de l'appel** (dans la fonction interne), pas au moment où tu construis l'outil, sinon tu figes la valeur du tour précédent. Si aucun code n'est disponible, renvoie `{'erreur': ...}` plutôt que de laisser un service deviner.
</details>

<details><summary>Aide +3 (pseudo-code)</summary>

```
def _substituer(code=None, bio_uniquement=False, limite=3):
    code = code ou session.code_confirme
    si code est vide : retourner {'erreur': 'aucun produit de référence confirmé'}
    retourner services.substituer({'code': code, 'bio_uniquement': ..., 'limite': ...})
```

`repondre_source` suit le même principe avec `services.rechercher_sources`.
</details>

In [ ]:
class ArgsChercherProduit(BaseModel):
    '''Arguments de chercher_produit (exemple complet fourni).'''
    model_config = ConfigDict(extra='forbid')
    nom: Optional[str] = Field(default=None, description='Nom ou mots du nom du produit (nom et marque possibles).')
    code: Optional[str] = Field(default=None, description='Code-barres du produit, en chaîne de caractères.')
    limite: StrictInt = Field(default=3, ge=1, le=5, description='Nombre maximum de produits renvoyés (1 à 5).')

    @model_validator(mode='after')
    def _verifier_nom_ou_code(self):
        if not (self.nom or '').strip() and not (self.code or '').strip():
            raise ValueError('il faut fournir un nom ou un code non vide')
        return self


class ArgsSubstituer(BaseModel):
    '''Arguments de substituer.'''
    model_config = ConfigDict(extra='forbid')
    # TODO 3a : ajoute les champs
    #   code : Optional[str], défaut None (complété depuis la session)
    #   bio_uniquement : StrictBool, défaut False
    #   limite : StrictInt entre 1 et 5, défaut 3
    # Écris une description claire pour chaque champ (le modèle la lit).


class ArgsRepondreSource(BaseModel):
    '''Arguments de repondre_source.'''
    model_config = ConfigDict(extra='forbid')
    # TODO 3b : ajoute les champs
    #   question : str non vide (min_length=1)
    #   code : Optional[str], défaut None (complété depuis la session)
    #   k : StrictInt entre 1 et 20, défaut 5


In [12]:
# Test 3 : les schémas refusent ce qu'ils doivent refuser
ArgsChercherProduit(code='123')  # accepté
for args in [{}, {'code': '1', 'inconnu': 1}, {'code': '1', 'limite': 0}, {'code': '1', 'limite': '3'}]:
    try:
        ArgsChercherProduit(**args)
        print('ACCEPTÉ (anormal)', args)
    except ValidationError:
        print('refusé (normal)', args)

assert ArgsSubstituer(bio_uniquement=True).limite == 3
assert ArgsSubstituer().code is None
for args in [{'limite': 6}, {'bio_uniquement': 'oui'}, {'champ_inconnu': 1}]:
    try:
        ArgsSubstituer(**args)
        raise AssertionError(f'ArgsSubstituer aurait dû refuser {args}')
    except ValidationError:
        pass

assert ArgsRepondreSource(question='Pourquoi ?').k == 5
for args in [{'question': ''}, {'question': 'x', 'k': 21}, {}]:
    try:
        ArgsRepondreSource(**args)
        raise AssertionError(f'ArgsRepondreSource aurait dû refuser {args}')
    except ValidationError:
        pass
print('Étape 3 (schémas) OK')

refusé (normal) {}
refusé (normal) {'code': '1', 'inconnu': 1}
refusé (normal) {'code': '1', 'limite': 0}
refusé (normal) {'code': '1', 'limite': '3'}


ValidationError: 1 validation error for ArgsSubstituer
bio_uniquement
  Extra inputs are not permitted [type=extra_forbidden, input_value=True, input_type=bool]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden

In [ ]:
def construire_outils(session: Session, services: Services) -> list:
    '''Construit les trois outils LangChain liés à une session et à des services.'''

    def _chercher(nom=None, code=None, limite=3):
        return services.chercher({'nom': nom, 'code': code, 'limite': limite})

    outil_chercher = StructuredTool.from_function(
        func=_chercher,
        name='chercher_produit',
        description=(
            '''Cherche un produit du catalogue par nom ou par code-barres et renvoie sa fiche '''
            '''(nutriments, Nutri-Score). À utiliser pour identifier un produit ou lire ses valeurs.'''
        ),
        args_schema=ArgsChercherProduit,
    )

    # TODO 3c : outil substituer (fonction interne _substituer + StructuredTool.from_function)
    #   - complète code avec session.code_confirme s'il est absent (lu AU MOMENT DE L'APPEL)
    #   - sans code disponible : renvoie {'erreur': ...}
    #   - description : quand l'utiliser ? quand mettre bio_uniquement=True ?

    # TODO 3d : outil repondre_source (même principe, services.rechercher_sources)
    #   - description : pour expliquer, justifier, citer des sources

    return [outil_chercher]  # TODO : ajoute les deux autres outils


In [ ]:
# Test 3 (outils) : les trois outils existent, et la session complète le code
_s = Session(session_id='test-outils', code_confirme=CODE_FIXTURE)
_outils = {o.name: o for o in construire_outils(_s, SERVICES_FIXTURE)}
assert set(_outils) == {'chercher_produit', 'substituer', 'repondre_source'}, list(_outils)

_res = _outils['substituer'].invoke({})
assert _res['statut'] == 'ok' and _res['code_reference'] == CODE_FIXTURE, _res

_s.code_confirme = None
_res = _outils['substituer'].invoke({})
assert 'erreur' in _res, _res   # pas de code : erreur claire, pas de devinette
print('Étape 3 (outils) OK')

## Étape 4 — Mémoire courte

On ne renvoie pas tout l'historique au modèle : on garde les **N derniers échanges complets**.

Un **échange** = un `HumanMessage` + tout ce qui suit jusqu'au prochain `HumanMessage` (appel d'outil `AIMessage`, `ToolMessage` associés, réponse finale).

Pourquoi « complets » ? Un `ToolMessage` sans l'`AIMessage` qui contient son appel (ou l'inverse) est rejeté par les API de modèles. Couper au milieu d'un échange casse la conversation.

Rappel : le code du produit confirmé n'est **pas** dans cette mémoire, il est dans `Session`. La troncature ne doit donc pas le faire perdre.

<details><summary>Aide +1 (question)</summary>

Si tu découpes en échanges, que reste-t-il à faire pour obtenir « les 3 derniers » ?
</details>

<details><summary>Aide +2 (approche)</summary>

Parcours la liste une fois en ouvrant un nouveau groupe à chaque `HumanMessage` (`isinstance`). Ensuite garde les `nb_echanges` derniers groupes et aplatis-les. Pense au cas d'une liste vide et aux messages éventuels avant le premier `HumanMessage`.
</details>

<details><summary>Aide +3 (pseudo-code)</summary>

```
echanges = []
pour chaque message :
    si c'est un HumanMessage : ouvrir un nouveau groupe
    ajouter le message au groupe courant (s'il existe)
retourner la concaténation des nb_echanges derniers groupes
```
</details>

In [ ]:
def decouper_en_echanges(messages: list) -> list:
    '''Découpe l'historique en échanges (liste de listes de messages).'''
    # TODO 4a
    raise NotImplementedError('TODO 4a')


def fenetre_messages(historique: list, nb_echanges: int = NB_ECHANGES_MEMOIRE) -> list:
    '''Renvoie les nb_echanges derniers échanges COMPLETS, à plat.'''
    # TODO 4b
    raise NotImplementedError('TODO 4b')


In [ ]:
# Test 4 : fenêtre de mémoire
def _echange(i, avec_outil=False):
    msgs = [HumanMessage(content=f'question {i}')]
    if avec_outil:
        msgs.append(AIMessage(content='', tool_calls=[{'name': 'chercher_produit', 'args': {'code': '1'}, 'id': f'call{i}'}]))
        msgs.append(ToolMessage(content='{}', tool_call_id=f'call{i}'))
    msgs.append(AIMessage(content=f'réponse {i}'))
    return msgs

_hist = []
for i in range(1, 6):
    _hist += _echange(i, avec_outil=(i % 2 == 1))

_fen = fenetre_messages(_hist, nb_echanges=3)
assert isinstance(_fen[0], HumanMessage) and _fen[0].content == 'question 3', _fen[0]
assert _fen[-1].content == 'réponse 5'
# chaque ToolMessage doit être précédé de l'AIMessage portant son appel
for idx, m in enumerate(_fen):
    if isinstance(m, ToolMessage):
        assert isinstance(_fen[idx - 1], AIMessage) and _fen[idx - 1].tool_calls
assert fenetre_messages([], 3) == []
assert len(fenetre_messages(_hist, 10)) == len(_hist)
print('Étape 4 OK')

## Étape 5 — Journalisation (traces)

À chaque échange (même sans appel d'outil), on écrit **une ligne JSON** (format JSONL) avec :

`session_id`, `tour`, `latence_ms`, `outils[]` (nom, statut, latence), `tokens_entree`, `tokens_sortie`, `statut`, `simule`.

Règle sur les tokens : on **additionne les usages des appels modèle réellement effectués**. Si l'usage n'est pas remonté (`usage_metadata` absent), on garde **None** (inconnu) : un faux `0` fausserait les coûts de la phase 7.

À décider et à documenter dans ta cellule : un tour **sans aucun appel modèle** vaut-il `(0, 0)` ou `(None, None)` ?

<details><summary>Aide +1 (question)</summary>

Un `AIMessage` renvoyé par LangChain possède un attribut `usage_metadata`. Que vaut-il quand le fournisseur ne remonte rien ?
</details>

<details><summary>Aide +2 (approche)</summary>

`usage_metadata` est un dictionnaire avec `input_tokens` et `output_tokens`, ou `None`. Si **un seul** message n'a pas d'usage, la somme serait fausse : renvoie `(None, None)`.

Pour écrire : crée le dossier parent (`mkdir(parents=True, exist_ok=True)`), ouvre en mode ajout, une ligne par trace, `ensure_ascii=False`.
</details>

<details><summary>Aide +3 (pseudo-code)</summary>

```
additionner_usage(messages_ia):
    si un message n'a pas usage_metadata : retourner (None, None)
    retourner (somme des input_tokens, somme des output_tokens)

ecrire_trace(trace, chemin=None):
    chemin = chemin ou FICHIER_TRACES      # lu à l'appel, pas à la définition
    créer le dossier, ajouter json.dumps(asdict(trace)) + saut de ligne
```
</details>

In [ ]:
@dataclass
class TraceOutil:
    nom: str
    statut: str
    latence_ms: float


@dataclass
class TraceEchange:
    session_id: str
    tour: int
    question: str
    latence_ms: float
    outils: list = field(default_factory=list)   # liste de dictionnaires (asdict de TraceOutil)
    tokens_entree: Optional[int] = None
    tokens_sortie: Optional[int] = None
    statut: str = 'ok'
    simule: bool = False


def additionner_usage(messages_ia: list) -> tuple:
    '''Renvoie (tokens_entree, tokens_sortie) cumulés, ou (None, None) si un usage manque.'''
    # TODO 5a
    raise NotImplementedError('TODO 5a')


def ecrire_trace(trace: TraceEchange, chemin: Optional[Path] = None) -> None:
    '''Ajoute la trace, en une ligne JSON, au fichier de traces.'''
    # TODO 5b
    raise NotImplementedError('TODO 5b')


In [ ]:
# Test 5 : agrégation des usages et écriture
_u1 = AIMessage(content='a', usage_metadata={'input_tokens': 10, 'output_tokens': 5, 'total_tokens': 15})
_u2 = AIMessage(content='b', usage_metadata={'input_tokens': 20, 'output_tokens': 7, 'total_tokens': 27})
assert additionner_usage([_u1, _u2]) == (30, 12)
assert additionner_usage([_u1, AIMessage(content='sans usage')]) == (None, None)

_chemin_test = Path('logs') / 'test_ecriture_trace.jsonl'
_chemin_test.unlink(missing_ok=True)
ecrire_trace(TraceEchange('s', 1, 'q', 12.5), chemin=_chemin_test)
ecrire_trace(TraceEchange('s', 2, 'q2', 3.0), chemin=_chemin_test)
_lignes = _chemin_test.read_text(encoding='utf-8').strip().splitlines()
assert len(_lignes) == 2 and json.loads(_lignes[1])['tour'] == 2
_chemin_test.unlink()
print('Étape 5 OK')

## Étape 6 — La boucle d'orchestration

Pour **un tour** de conversation :

1. construire les messages : prompt système + mémoire courte + question ;
2. appeler le modèle (avec les outils liés) ;
3. si le modèle demande des outils : les exécuter, renvoyer les résultats (`ToolMessage`), puis rappeler le modèle ;
4. s'arrêter quand le modèle répond sans appel d'outil, ou après `MAX_ITERATIONS_OUTILS` ;
5. écrire la trace, mettre à jour l'historique.

`jouer_tour` est fourni. Tu dois écrire ses deux fonctions auxiliaires :

- **TODO 6a `executer_appel`** : exécute un appel d'outil, mesure sa latence, **ne lève jamais d'exception** ;
- **TODO 6b `mettre_a_jour_code_confirme`** : décide quand le produit de référence change.

Lis aussi le prompt système : c'est un point de départ, pas une version finale (garde-fous au TP22).

<details><summary>Aide +1 (6a — question)</summary>

Qu'est-ce qui peut mal tourner quand on exécute un outil ? Arguments invalides, outil inconnu, service en panne. Le modèle doit-il recevoir une exception Python ou un résultat qu'il peut lire ?
</details>

<details><summary>Aide +2 (6a — approche)</summary>

`appel` est un dictionnaire `{'name', 'args', 'id'}`. Mesure avec `time.perf_counter()`. Trois cas d'erreur distincts, chacun avec son statut de trace : `outil_inconnu`, `erreur_arguments` (`ValidationError`), `erreur_service` (autre exception). En cas de succès, le statut de trace est `resultat.get('statut')`. Dans tous les cas d'erreur le résultat est `{'erreur': message}`.
</details>

<details><summary>Aide +3 (6a — pseudo-code)</summary>

```
debut = perf_counter()
outil = outils_par_nom.get(appel['name'])
si outil absent : resultat = {'erreur': ...}, statut = 'outil_inconnu'
sinon essayer : resultat = outil.invoke(appel['args'])
    sauf ValidationError : ... 'erreur_arguments'
    sauf Exception : ... 'erreur_service'
latence_ms = (perf_counter() - debut) * 1000
retourner resultat, TraceOutil(nom, statut, latence_ms)
```
</details>

<details><summary>Aide +1 (6b — question)</summary>

Dans quel cas précis sait-on **avec certitude** de quel produit on parle ? Et si la recherche est ambiguë ?
</details>

<details><summary>Aide +2 (6b — approche)</summary>

Seul `chercher_produit` peut changer le code confirmé, et seulement si `statut == 'ok'` et `total == 1`. Dans tous les autres cas (ambigu, introuvable, erreur, autres outils) le code reste inchangé. Le code est dans `resultat['produits'][0]['code']`.
</details>

In [ ]:
PROMPT_SYSTEME = '''Tu es l'assistant NutriScope, spécialisé dans la qualité nutritionnelle des produits alimentaires.
Règles :
- Pour toute information sur un produit, appelle un outil. N'invente jamais une valeur.
- chercher_produit : identifier un produit et lire sa fiche.
- repondre_source : expliquer, justifier, citer des sources.
- substituer : proposer un produit mieux noté dans le même rayon. Utilise bio_uniquement=True seulement si l'utilisateur le demande explicitement.
- Si la recherche est ambiguë, présente les candidats et demande de préciser. Ne choisis pas à sa place.
- Si une valeur est inconnue (null), dis que l'information manque.
- Cite les sources renvoyées par repondre_source.
- Pas de conseil médical. Reste dans le domaine alimentaire.
Réponds en français, de façon concise.'''


def executer_appel(appel: dict, outils_par_nom: dict) -> tuple:
    '''Exécute un appel d'outil. Renvoie (resultat: dict, trace_outil: TraceOutil).
    Ne lève jamais d'exception : les erreurs deviennent un résultat {'erreur': ...}.'''
    # TODO 6a
    raise NotImplementedError('TODO 6a')


def mettre_a_jour_code_confirme(session: Session, nom_outil: str, resultat: dict) -> None:
    '''Met à jour session.code_confirme uniquement quand le produit est certain.'''
    # TODO 6b
    raise NotImplementedError('TODO 6b')


def jouer_tour(session: Session, question: str, llm, services: Services = SERVICES_FIXTURE) -> dict:
    '''Joue un tour de conversation. Renvoie {'reponse': str, 'trace': dict}.'''
    debut = time.perf_counter()
    session.numero_tour += 1
    outils = construire_outils(session, services)
    outils_par_nom = {o.name: o for o in outils}
    llm_outille = llm.bind_tools(outils)

    message_question = HumanMessage(content=question)
    messages = [SystemMessage(content=PROMPT_SYSTEME)]
    messages += fenetre_messages(session.historique)
    messages.append(message_question)

    nouveaux = [message_question]   # ce qui s'ajoutera à l'historique à la fin du tour
    messages_ia, traces_outils = [], []
    statut = 'limite_outils_atteinte'

    for _ in range(MAX_ITERATIONS_OUTILS):
        reponse = llm_outille.invoke(messages)
        messages.append(reponse)
        nouveaux.append(reponse)
        messages_ia.append(reponse)
        if not reponse.tool_calls:
            statut = 'ok'
            break
        for appel in reponse.tool_calls:
            resultat, trace_outil = executer_appel(appel, outils_par_nom)
            mettre_a_jour_code_confirme(session, appel['name'], resultat)
            message_outil = ToolMessage(content=json.dumps(resultat, ensure_ascii=False), tool_call_id=appel['id'])
            messages.append(message_outil)
            nouveaux.append(message_outil)
            traces_outils.append(trace_outil)

    if statut != 'ok':
        nouveaux.append(AIMessage(content='Je ne parviens pas à terminer cette demande.'))

    tokens_entree, tokens_sortie = additionner_usage(messages_ia)
    trace = TraceEchange(
        session_id=session.session_id, tour=session.numero_tour, question=question,
        latence_ms=round((time.perf_counter() - debut) * 1000, 1),
        outils=[asdict(t) for t in traces_outils],
        tokens_entree=tokens_entree, tokens_sortie=tokens_sortie,
        statut=statut, simule=services.simule,
    )
    ecrire_trace(trace)
    session.historique.extend(nouveaux)
    return {'reponse': nouveaux[-1].content, 'trace': asdict(trace)}


## Étape 7 — Tester la plomberie sans modèle

Avant de brancher un vrai modèle (lent, non déterministe), on teste l'orchestration avec un **modèle scripté** : il rejoue des réponses préparées, dont les appels d'outils attendus. Ça valide : exécution des outils, injection du code depuis la session, mémoire, traces.

Ça **ne valide pas** que le vrai modèle choisit le bon outil : c'est l'étape 10.

**Exercice 1 / 2.** Lance la cellule de test. Si une assertion échoue, remonte à l'étape concernée (3c, 3d, 4, 5, 6a, 6b).

**Question à te poser** : pourquoi, au tour 4, `substituer` reçoit-il `bio_uniquement=True` mais pas de `code` ? D'où vient le code ?

In [ ]:
def scenario_tp21(code_produit: str) -> list:
    '''Décrit les entrées et les vérifications du scénario d'acceptation.'''
    return [
        {'question': f'Ce produit est-il bon ? Son code-barres est {code_produit}.',
         'outil_attendu': 'chercher_produit',
         'controle': 'Fiche du code confirmé et valeurs concordantes avec le catalogue.'},
        {'question': 'Pourquoi ?',
         'outil_attendu': 'repondre_source',
         'controle': 'Même produit, explication étayée et identifiants de sources vérifiables.'},
        {'question': 'Propose mieux.',
         'outil_attendu': 'substituer',
         'controle': 'Même référence, produit comparable et mieux noté.'},
        {'question': 'Et en bio ?',
         'outil_attendu': 'substituer',
         'controle': 'Même référence, bio_uniquement=True et bio confirmé.'},
    ]


class LLMScripte:
    '''Faux modèle : rejoue des AIMessage préparés. Sert uniquement à tester l'orchestration.'''
    def __init__(self, reponses):
        self.reponses = list(reponses)

    def bind_tools(self, outils):
        return self

    def invoke(self, messages):
        return self.reponses.pop(0)


def _appel_ia(nom, args, identifiant):
    return AIMessage(content='', tool_calls=[{'name': nom, 'args': args, 'id': identifiant}],
                     usage_metadata={'input_tokens': 10, 'output_tokens': 5, 'total_tokens': 15})


def _texte_ia(texte):
    return AIMessage(content=texte,
                     usage_metadata={'input_tokens': 10, 'output_tokens': 5, 'total_tokens': 15})


In [ ]:
# Test 7 : scénario complet, modèle scripté + fixtures
_ancien_fichier = FICHIER_TRACES
FICHIER_TRACES = Path('logs') / 'traces_tp21_test.jsonl'
FICHIER_TRACES.unlink(missing_ok=True)

_reponses = [
    _appel_ia('chercher_produit', {'code': CODE_FIXTURE}, 'c1'), _texte_ia('Fiche trouvée.'),
    _appel_ia('repondre_source', {'question': 'Pourquoi ?'}, 'c2'), _texte_ia('Explication.'),
    _appel_ia('substituer', {}, 'c3'), _texte_ia('Voici un meilleur produit.'),
    _appel_ia('substituer', {'bio_uniquement': True}, 'c4'), _texte_ia('Voici une option bio.'),
]
_session = Session(session_id='test-routage')
_llm = LLMScripte(_reponses)
_sorties = [jouer_tour(_session, t['question'], _llm) for t in scenario_tp21(CODE_FIXTURE)]

_outils_utilises = [[o['nom'] for o in s['trace']['outils']] for s in _sorties]
print(_outils_utilises)
assert _outils_utilises == [['chercher_produit'], ['repondre_source'], ['substituer'], ['substituer']]
assert _session.code_confirme == CODE_FIXTURE            # tour 1 : produit confirmé

_resultats = [json.loads(m.content) for m in _session.historique if isinstance(m, ToolMessage)]
assert _resultats[1]['statut'] == 'ok'                    # tour 2 : code repris de la session
assert _resultats[2]['substituts'][0]['produit']['code'] == '9990000000004'  # meilleur grade
assert all(s['bio'] is True for s in _resultats[3]['substituts'])            # tour 4 : filtre bio
assert all(s['trace']['tokens_entree'] == 20 for s in _sorties)              # 2 appels modèle par tour
assert len(FICHIER_TRACES.read_text(encoding='utf-8').strip().splitlines()) == 4

FICHIER_TRACES.unlink()
FICHIER_TRACES = _ancien_fichier
print('Étape 7 OK : la plomberie fonctionne (routage simulé uniquement)')

## Étape 8 — Scénario rejouable et rapport

Le livrable demande des **scénarios rejouables** : une fonction qui, pour un modèle, des services et un code produit donnés, rejoue les 4 tours et produit un **rapport** comparant l'outil attendu à l'outil réellement appelé.

Chaque ligne du rapport doit contenir : tour, question, outil attendu, outils appelés, latence, tokens, statut, et un booléen `routage_ok`. Le rapport doit indiquer si les **services sont simulés**.

<details><summary>Aide +1 (question)</summary>

Quelle valeur de `routage_ok` pour un tour où le modèle appelle deux outils dont le bon ? Décide-le et justifie-le dans un commentaire.
</details>

<details><summary>Aide +2 (approche)</summary>

Crée une `Session` neuve, parcours `scenario_tp21(code_produit)`, appelle `jouer_tour`, compare `outil_attendu` à la liste des noms d'outils de `trace['outils']`. Ne mélange pas les traces d'un rejeu avec celles d'un autre : utilise un `session_id` distinct.
</details>

In [ ]:
def rejouer_scenario(llm, services: Services, code_produit: str, session_id: str = 'rejeu') -> list:
    '''Rejoue les 4 tours du scénario et renvoie une liste de lignes de rapport (dictionnaires).'''
    # TODO 8a
    raise NotImplementedError('TODO 8a')


def afficher_rapport(rapport: list, simule: bool) -> None:
    print('⚠️ SERVICES SIMULÉS : routage seulement' if simule else 'Services réels')
    for ligne in rapport:
        marque = 'OK ' if ligne['routage_ok'] else 'KO '
        print(f"{marque} tour {ligne['tour']} | attendu={ligne['outil_attendu']} | appelés={ligne['outils_appeles']} "
              f"| {ligne['latence_ms']} ms | tokens={ligne['tokens_entree']}/{ligne['tokens_sortie']} | {ligne['statut']}")


In [ ]:
# Test 8 : le rapport est cohérent avec le modèle scripté
_ancien_fichier = FICHIER_TRACES
FICHIER_TRACES = Path('logs') / 'traces_tp21_test.jsonl'
_rapport = rejouer_scenario(LLMScripte(_reponses_8 := [
    _appel_ia('chercher_produit', {'code': CODE_FIXTURE}, 'c1'), _texte_ia('Fiche trouvée.'),
    _appel_ia('repondre_source', {'question': 'Pourquoi ?'}, 'c2'), _texte_ia('Explication.'),
    _appel_ia('substituer', {}, 'c3'), _texte_ia('Mieux.'),
    _appel_ia('substituer', {'bio_uniquement': True}, 'c4'), _texte_ia('Bio.'),
]), SERVICES_FIXTURE, CODE_FIXTURE, session_id='test-rapport')
afficher_rapport(_rapport, simule=True)
assert len(_rapport) == 4 and all(l['routage_ok'] for l in _rapport)
FICHIER_TRACES.unlink(missing_ok=True)
FICHIER_TRACES = _ancien_fichier
print('Étape 8 OK')

## Étape 9 — Brancher les services réels

Il faut maintenant écrire trois adaptateurs qui respectent **exactement** le même contrat que les fixtures (dictionnaire en entrée, dictionnaire en sortie).

**Avant de coder, lis les diffs et pose-toi ces questions** (je ne connais pas le contenu exact de ton dépôt : vérifie-le) :

| Service | À vérifier dans ton dépôt |
|---|---|
| `chercher` | Quelle fonction du TP18 (PR #71 : `chercher_produit(arguments, catalogue=...)`) ? Quel objet catalogue lui passer ? Les inconnues restent-elles à `null` ? |
| `substituer` | Où est ton `src/substitution.py` du TP16 ? Quelle signature ? Le lot de données expose-t-il **rayon, sous-catégorie, statut bio** ? (ton analyse notait que le lot #71 n'expose ni sous-catégorie ni bio : il faudra préparer une projection) |
| `repondre_source` | `rechercher(question, k=..., filtres=...)` du module `src/rag` (contrat : `id, score, texte, metadonnees`). Quelles métadonnées sont **réellement indexées** pour filtrer ? |

### `repondre_source` : retrieval, contexte, génération, vérification

1. **Retrieval** : récupère les passages (`rechercher`). Pour « pourquoi ? », joins la fiche du code confirmé **et** les passages généraux (ne filtre pas tout par code, sinon les passages généraux disparaissent).
2. **Contexte** : assemble fiche + passages, chacun avec son identifiant.
3. **Génération** : réponse construite à partir de ces preuves uniquement.
4. **Vérification** : les identifiants cités doivent appartenir aux sources réellement récupérées (fonction `verifier_citations`). Un identifiant inventé est rejeté.
5. Pas assez de preuves : statut `sources_insuffisantes` et explication, plutôt qu'une réponse inventée.

Pense aussi au **grade périmé** : si la fiche indexée a un grade et que le catalogue actuel l'a passé à inconnu, quelle source fait foi ?

<details><summary>Aide +1 (question)</summary>

Si le service de retrieval est indisponible (Chroma non démarré, index vide), que doit voir le modèle : une exception, un résultat vide ou une erreur explicite ?
</details>

<details><summary>Aide +2 (approche)</summary>

Distingue trois situations : index vide ou aucun passage pertinent → `sources_insuffisantes` ; service en panne → erreur distincte (`{'erreur': ...}`, levée côté `executer_appel`) ; succès → `ok`. `verifier_citations` est une simple comparaison d'ensembles.
</details>

In [ ]:
# TODO 9a : imports réels. Remplace les chemins par ceux de TON dépôt après lecture des diffs.
# from src.rag.recherche import rechercher
# from src.assistant.<module> import chercher_produit as chercher_produit_reel


def chercher_reel(arguments: dict) -> dict:
    '''Adaptateur vers le vrai chercher_produit. Même contrat que chercher_fixture.'''
    # TODO 9b
    raise NotImplementedError('TODO 9b')


def substituer_reel(arguments: dict) -> dict:
    '''Adaptateur vers le moteur de substitution du TP16. Même contrat que substituer_fixture.'''
    # TODO 9c
    raise NotImplementedError('TODO 9c')


def verifier_citations(ids_cites: list, sources: list) -> list:
    '''Renvoie la liste des identifiants cités qui ne figurent PAS dans les sources récupérées.'''
    # TODO 9d
    raise NotImplementedError('TODO 9d')


def repondre_source_reel(arguments: dict) -> dict:
    '''Retrieval + contexte + génération + vérification des citations. Même contrat que la fixture.'''
    # TODO 9e
    raise NotImplementedError('TODO 9e')


SERVICES_REELS = Services(chercher_reel, substituer_reel, repondre_source_reel, simule=False)


## Étape 10 — Rejeu avec le vrai modèle

Deux paliers :

1. **Vrai modèle + fixtures** : vérifie que **le modèle choisit le bon outil** (routage réel) sans dépendre des services réels. Si le routage échoue (par exemple `substituer` au lieu de `repondre_source` pour « Pourquoi ? »), améliore les **descriptions d'outils** et le **prompt système**, pas le code de la boucle.
2. **Vrai modèle + services réels** : le scénario d'acceptation. Remplis `CODE_REEL` avec un code présent dans ton lot pilote. Avant de lancer, vérifie que ce lot permet au moins un meilleur candidat comparable, une option bio, et que le corpus contient une référence exploitable.

Lance Ollama avant (`ollama serve` si besoin) et vérifie que le modèle est installé.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODELE, base_url=URL_OLLAMA, temperature=0)

# Palier 1 : vrai modèle, services fictifs (routage réel)
rapport_routage = rejouer_scenario(llm, SERVICES_FIXTURE, CODE_FIXTURE, session_id='rejeu-routage')
afficher_rapport(rapport_routage, simule=True)

In [ ]:
# Palier 2 : vrai modèle, services réels
CODE_REEL = None   # TODO : code-barres (chaîne) présent dans le lot pilote

if CODE_REEL is not None:
    rapport_reel = rejouer_scenario(llm, SERVICES_REELS, CODE_REEL, session_id='rejeu-reel')
    afficher_rapport(rapport_reel, simule=False)
else:
    print('Renseigne CODE_REEL puis relance cette cellule.')

### Lecture des traces (à faire avec le vrai rejeu)

Ouvre `FICHIER_TRACES` et vérifie, pour chaque échange : l'outil appelé est-il le bon ? la latence est-elle plausible ? les tokens sont-ils renseignés ou `null` ? Les sources citées existent-elles réellement dans les passages récupérés ?

Note ici 3 à 5 lignes de constats honnêtes (échecs de routage, latence de l'outil le plus lent, tokens manquants) : ce sont ces mesures dont tu auras besoin en phase 7.

In [ ]:
# TODO 10 : lis le fichier de traces (une ligne JSON par échange) et affiche un résumé :
#   latence moyenne par tour, latence par outil, total de tokens (en excluant les None).
# Astuce : json.loads sur chaque ligne ; pandas.DataFrame(lignes) si tu veux un tableau.


## Étape 11 — Cas de contrôle

Rejoue chacun de ces cas (avec modèle scripté puis vrai modèle quand c'est pertinent) et coche quand c'est vérifié.

| Cas | Attendu | Vérifié |
|---|---|---|
| Recherche ambiguë, puis « Pourquoi ? » | Demander de préciser le produit. Aucun code arbitraire dans la session. | ☐ |
| Grade ou valeur nutritionnelle inconnue | `null` conservé dans la fiche ; l'assistant explique que l'information manque. | ☐ |
| Aucun meilleur candidat bio | Liste vide et message explicite, pas d'élargissement silencieux. | ☐ |
| Index vide, source absente ou identifiant inventé | `sources_insuffisantes` ou rejet de la citation. | ☐ |
| Nouvelle session après le premier parcours | « Et en bio ? » demande un produit de référence. | ☐ |
| Historique tronqué (plus de 3 échanges) | Appels et résultats restent appariés ; le code confirmé reste accessible via la session. | ☐ |
| Tour sans usage de tokens remonté | Jetons signalés `None`, pas `0`. | ☐ |
| Arguments invalides envoyés par le modèle | Résultat `{erreur}` renvoyé au modèle, pas d'exception. | ☐ |

**Exercice bonus** : écris 2 ou 3 de ces cas sous forme de tests `pytest` dans `tests/` (le projet utilise déjà pytest). Les cas ambigus et de session neuve se testent très bien avec `LLMScripte` et les fixtures.

## Étape 12 — Revue croisée et rendu

### Revue croisée (10 minutes)

Conversation libre avec l'assistant d'une autre équipe, puis l'inverse. Fais-lui demander : une explication, un substitut, une variante bio, puis essaie une ambiguïté et une information absente.

Pour chaque anomalie, consigne dans une **issue GitHub** pendant la séance :

- la question exacte ;
- l'état de départ (session neuve ? produit déjà confirmé ?) ;
- le résultat attendu ;
- le résultat observé ;
- la référence de la trace (`session_id` + `tour`).

### Checklist de rendu

- [ ] Assistant LangChain, mémoire courte et trois outils réels.
- [ ] Scénarios rejouables, avec résultats et provenance des sources ; parcours simulés identifiés comme tels.
- [ ] Trace active à chaque échange (outil, latence, tokens).
- [ ] Revue croisée effectuée et remarques consignées en issues.

### Organisation Git proposée (selon ton `CONTRIBUTING.md`)

Branche : `<Prénom>/feat/tp21_orchestration`, partie de `develop`, PR relue par ton binôme. Commits courts en français, sans préfixe de type, un changement logique par commit, par exemple :

```
ajoute la session et les schémas des trois outils
ajoute la mémoire courte par échanges complets
ajoute la journalisation des échanges en JSONL
ajoute la boucle d'orchestration et les tests avec modèle scripté
branche les services réels dans le scénario rejouable
```

Pense à documenter dans `docs/journal.md` (3 lignes : fait / décidé / bloqué) et à noter les choix défendables à l'oral : boucle d'outils manuelle, code confirmé dans la session, tokens `None` plutôt que `0`, fixtures séparées du réel.